# 05 · Results Analysis

Score each GPT model against the manually annotated ground truth (thesis §4): build true-vs-predicted comparison tables, plot confusion matrices, and report per-dimension accuracy.

The raw model outputs contain noisy / off-vocabulary labels. The hand-curated fixes for each model live as data in `data/processed/corrections/` and are applied via `apply_label_corrections`.

**Accuracy metric.** Following the thesis, per-dimension "accuracy" is the **macro-averaged per-class recall** (the unweighted mean of the four per-class recalls in Tables 1a/1b), *not* the raw fraction of companies correct. Macro-averaging is the appropriate choice for these imbalanced classes.


In [ ]:
import pandas as pd

from company_assessment import config
from company_assessment.evaluation import (
    apply_label_corrections,
    build_comparison_df,
    accuracy_report,
    per_class_recall,
    plot_comparison_confusion_matrices,
)

## Ground truth

Drop the few-shot example companies (they were shown to the model) and one stray duplicate (`BeGo`).

In [ ]:
labels = pd.read_csv(config.PROCESSED_DIR / 'final.csv')
examples_df = pd.read_csv(config.EXAMPLES_CSV)

example_ids = examples_df['company_id'].tolist()
labels_filtered = labels[~labels['company_id'].isin(example_ids)]
labels_filtered = labels_filtered[labels_filtered['company_name'] != 'BeGo']
true_labels = labels_filtered[['company_id', 'UVP', 'Data Uniqueness']]
true_labels.shape

## Per-model evaluation

Each model follows the same three steps: apply its manual corrections, build the comparison frame, then report accuracy and plot the confusion matrices. `comparison_df_*.csv` files are written to `data/processed/`.

In [ ]:
MODELS = {
    'gpt3_5':      ('final_classified_gpt3turbo.csv',      'GPT-3.5-turbo'),
    'gpt4o':       ('final_classified_gpt4o.csv',          'GPT-4o'),
    'gpt-4-turbo': ('final_classified_gpt-4-turbo.csv',    'GPT-4-turbo'),
}

comparisons = {}
for key, (filename, label) in MODELS.items():
    results = pd.read_csv(config.PROCESSED_DIR / filename)
    results = results[results['company_name'] != 'BeGo']
    results = apply_label_corrections(results, key)

    comparison = build_comparison_df(true_labels, results)
    comparisons[key] = comparison
    comparison.to_csv(config.PROCESSED_DIR / f'comparison_df_{key}.csv', index=False)

    report = accuracy_report(comparison)
    print(f"{label:>14} | UVP {report['UVP']:.3f} | "
          f"Data Uniqueness {report['Data Uniqueness']:.3f} | "
          f"overall {report['overall']:.3f}")

### Confusion matrices

In [ ]:
for key, (_, label) in MODELS.items():
    plot_comparison_confusion_matrices(comparisons[key], title_prefix=f'{label}: ')

## Summary table

In [ ]:
summary = pd.DataFrame(
    {label: accuracy_report(comparisons[key]) for key, (_, label) in MODELS.items()}
).T
summary

## Per-class recall (thesis Tables 1a / 1b)

The per-class accuracy behind each dimension's macro-average.

In [ ]:
for dim in ('UVP', 'Data Uniqueness'):
    print(f'\n=== {dim} per-class recall ===')
    table = pd.DataFrame(
        {label: per_class_recall(comparisons[key], dim)
         for key, (_, label) in MODELS.items()}
    )
    display(table.round(4))